In [11]:
import sys
import types
import torchvision.transforms.functional as F

# Buat alias lama
sys.modules['torchvision.transforms.functional_tensor'] = F


In [12]:
import os
import cv2
import torch
from tqdm import tqdm
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

In [7]:
# --- STEP 1: MODEL SETUP ---
# initialize the pre-trained Real-ESRGAN model. This is done only once.

def setup_super_resolution_model():
    """
    Downloads (if needed) and sets up the Real-ESRGAN model.
    This function initializes the upsampler with a powerful pre-trained model
    capable of general-purpose 4x upscaling.

    Returns:
        RealESRGANer: An upsampler object ready to process images.
    """
    print("Setting up the Super-Resolution model...")

    # Define the model parameters. We are using RealESRGAN_x4plus, a robust model for general content.
    model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
    netscale = 4
    model_path = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'

    # Determine the device to run on (GPU if available, otherwise CPU)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Model will run on: {device}")

    # Initialize the upsampler.
    # 'half=True' enables half-precision (FP16) for faster processing on compatible GPUs.
    upsampler = RealESRGANer(
        scale=netscale,
        model_path=model_path,
        model=model,
        tile=0,
        tile_pad=10,
        pre_pad=0,
        half=torch.cuda.is_available(), # Enable half-precision only if a CUDA GPU is available
        device=device
    )

    print("Model setup complete.")
    return upsampler

In [8]:
# --- STEP 2: VIDEO PROCESSING FUNCTION ---
# This function contains the core logic for upscaling the video. It reads
# the input video frame by frame, applies the super-resolution model to each
# frame, and writes the upscaled frames to a new output video file.

def upscale_video(input_path, output_path, upsampler_model):
    """
    Reads a video from input_path, upscales each frame using the provided
    upsampler_model, and saves the result to output_path.
    """
    try:
        # Open the input video file for reading
        cap = cv2.VideoCapture(input_path)
        if not cap.isOpened():
            print(f"Error: Could not open input video file at {input_path}")
            return

        # Get original video properties
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        # Define the output video properties (4x resolution)
        new_width, new_height = width * 4, height * 4

        # Setup the video writer object to save the output video
        # We use the 'mp4v' codec for MP4 files.
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, fps, (new_width, new_height))

        print("\n" + "="*50)
        print(f"Processing Video: {os.path.basename(input_path)}")
        print(f"  - Original Resolution: {width}x{height}")
        print(f"  - Target Resolution:   {new_width}x{new_height}")
        print(f"  - Frame Rate (FPS):    {fps:.2f}")
        print(f"  - Total Frames:        {frame_count}")
        print("="*50 + "\n")

        # Loop through each frame of the video
        for _ in tqdm(range(frame_count), desc="Upscaling Frames", unit="frame"):
            ret, frame = cap.read()
            if not ret:
                # Break the loop if there are no more frames to read
                break

            # Use the upsampler to enhance the frame resolution.
            # The model expects BGR format, which is the default for OpenCV.
            output_frame, _ = upsampler_model.enhance(frame, outscale=4)

            # Write the enhanced frame to the output video file
            out.write(output_frame)

        # Release the video capture and writer objects to free up resources
        cap.release()
        out.release()

        print("\n" + "="*50)
        print("SUCCESS!")
        print(f"Video has been upscaled and saved to: {output_path}")
        print("="*50)

    except Exception as e:
        print(f"\nAn error occurred during video processing: {e}")

In [9]:
# --- STEP 4: DIRECTORY PROCESSING FUNCTION ---
# This new function iterates through all subdirectories of a root folder,
# finds all video files, and processes them.

def process_directory(input_dir, output_dir, upsampler_model):
    """
    Recursively finds all video files in the input_dir, upscales them,
    and saves them to the output_dir, preserving the directory structure.
    """
    video_extensions = ('.mp4', '.avi', '.mov', '.mkv', '.flv', '.wmv')

    # Walk through the directory tree
    for root, _, files in os.walk(input_dir):
        for filename in files:
            # Check if the file is a video
            if filename.lower().endswith(video_extensions):
                input_path = os.path.join(root, filename)

                # Create the corresponding output directory structure
                # This mirrors the folder hierarchy from the input directory
                relative_path = os.path.relpath(root, input_dir)
                output_subdir = os.path.join(output_dir, relative_path)
                os.makedirs(output_subdir, exist_ok=True)

                # Define the output file path with an '_upscaled' suffix
                base, ext = os.path.splitext(filename)
                output_path = os.path.join(output_subdir, f"{base}_upscaled{ext}")

                # A helpful feature: skip if the upscaled file already exists
                if os.path.exists(output_path):
                    print(f"Skipping already processed file: {output_path}")
                    continue

                # Process the individual video file
                upscale_video(input_path, output_path, upsampler_model)

In [10]:
input_root_dir = ''
output_root_dir = ''

if os.path.isdir(input_root_dir):
    # 1. Set up the super-resolution model
    upsampler = setup_super_resolution_model()

    # 2. Process all videos in the specified directory
    process_directory(input_root_dir, output_root_dir, upsampler)
else:
    print(f"Error: Input directory not found at '{input_root_dir}'.")
    print("Please make sure the path is correct.")

Error: Input directory not found at ''.
Please make sure the path is correct.
